##Mount Drive & Paths

In [ ]:
# @title Mount Google Drive & set paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
RAW_DIR = Path("/content/drive/My Drive/PropInsight/raw/data_gov")
PREP_DIR = Path("/content/drive/My Drive/PropInsight/preprocess/data_gov")
PREP_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR)
print("PREP_DIR:", PREP_DIR)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RAW_DIR: /content/drive/My Drive/PropInsight/raw/data_gov
PREP_DIR: /content/drive/My Drive/PropInsight/preprocess/data_gov


In [ ]:
from pathlib import Path

# First, check if Google Drive is mounted
drive_root = Path("/content/drive")
print(f"Drive root exists? {drive_root.exists()}")

if drive_root.exists():
    print(f"✅ Google Drive is mounted")

    # Check MyDrive
    my_drive = Path("/content/drive/MyDrive")
    print(f"MyDrive exists? {my_drive.exists()}")

    if my_drive.exists():
        # List contents to verify
        print("\n📁 Contents of MyDrive:")
        for item in sorted(my_drive.iterdir())[:10]:
            print(f"  - {item.name}")

    # Check PropInsight
    prop_insight = Path("/content/drive/MyDrive/PropInsight")
    print(f"\nPropInsight exists? {prop_insight.exists()}")

    if prop_insight.exists():
        print("📁 Contents of PropInsight:")
        for item in sorted(prop_insight.iterdir()):
            print(f"  - {item.name}")

        # Check raw folder
        raw_folder = prop_insight / "raw"
        if raw_folder.exists():
            print("\n📁 Contents of raw/:")
            for item in sorted(raw_folder.iterdir()):
                print(f"  - {item.name}")

            # Check data_gov specifically
            data_gov = raw_folder / "data_gov"
            print(f"\ndata_gov exists? {data_gov.exists()}")
            if data_gov.exists():
                print("📁 Contents of raw/data_gov/:")
                for item in sorted(data_gov.iterdir())[:20]:
                    print(f"  - {item.name}")
else:
    print("❌ Google Drive is NOT mounted!")
    print("\nRun this cell to mount Google Drive:")
    print("---")
    print("from google.colab import drive")
    print("drive.mount('/content/drive')")


Drive root exists? True
✅ Google Drive is mounted
MyDrive exists? True

📁 Contents of MyDrive:
  - 002.mp4
  - 18_08_2025_Recording.wav
  - 20190906_140942 (1).jpg
  - 20190906_140942.jpg
  - 2023-Sample-ChatGPT.gdoc
  - 2024-08-03 10-32-55.mp4
  - 2212.10156v2.pdf
  - 2306.16927v3.pdf
  - 2401.08658v1.pdf
  - 2405.19620v2.pdf

PropInsight exists? True
📁 Contents of PropInsight:
  - labeled
  - preprocess
  - processed
  - raw

📁 Contents of raw/:
  - data_gov
  - forums

data_gov exists? True
📁 Contents of raw/data_gov/:
  - BookingsforNewFlats.csv
  - DemandforRentalandSoldFlats.csv
  - ExecutiveCondominiumUnitsLaunchedandSoldintheQuarterQuarterly.csv
  - PrivateResidentialPropertyPriceIndexBaseQuarter2009Q1100.csv
  - PrivateResidentialPropertyPriceIndexBaseQuarter2009Q1100Quarterly.csv
  - PrivateResidentialPropertyRentalIndexBaseQuarter2009Q1100Quarterly.csv
  - PrivateResidentialPropertyTransactionsinCoreCentralRegionQuarterly.csv
  - PrivateResidentialPropertyTransactionsinOutsi

#Utilities (parsers, numeric coercion, notes, postprocess_common)

In [ ]:
# @title Utilities: parsing, normalization, guards, metadata
import re, json
import pandas as pd
import numpy as np
from pandas.tseries.offsets import MonthBegin
from pathlib import Path

# ---------- Column normalization ----------
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    return df

# ---------- Quarter parsing ----------
# Accepts 'YYYYQn', 'YYYY-Qn', 'YYYY Qn', 'Qn YYYY'
QUARTER_RE = re.compile(r"^(?P<year>20[0-9]{2})\D*Q(?P<q>[1-4])$", re.IGNORECASE)

def detect_quarter_headers(df):
    out = []
    for c in df.columns:
        m = QUARTER_RE.match(str(c).strip())
        if m:
            out.append((c, int(m.group("year")), int(m.group("q"))))
    return out

def parse_quarter_str(val):
    if pd.isna(val): return (None, None, None)
    s = str(val).strip()
    m = re.search(r"(20\d{2}).*?Q([1-4])", s, re.IGNORECASE)
    if not m:
        m = re.search(r"Q([1-4]).*?(20\d{2})", s, re.IGNORECASE)
        if m:
            return (f"{int(m.group(2))}-Q{int(m.group(1))}", int(m.group(2)), int(m.group(1)))
        return (None, None, None)
    year, q = int(m.group(1)), int(m.group(2))
    return (f"{year}-Q{q}", year, q)

# ---------- Numeric coercion ----------
def coerce_numeric(series, allow_float=True):
    s = series.astype(str).str.replace(",", "", regex=False).str.strip()
    num = pd.to_numeric(s, errors="coerce")
    if not allow_float:
        num = num.astype("Int64")
    return num

# ---------- Unicode & whitespace cleanup ----------
def strip_invisible_chars(df):
    for c in df.columns:
        if pd.api.types.is_string_dtype(df[c]):
            df[c] = (
                df[c]
                .astype(str)
                .str.replace(r"[\u200b\u200c\u200d\uFEFF\u00A0]", "", regex=True)
                .str.strip()
            )
    return df

# ---------- Remaining lease / Storey parsing ----------
def parse_remaining_lease_to_months(val):
    if pd.isna(val): return np.nan
    s = str(val)
    y = re.search(r"(\d+)\s*year", s)
    m = re.search(r"(\d+)\s*month", s)
    yrs = int(y.group(1)) if y else 0
    mos = int(m.group(1)) if m else 0
    return yrs * 12 + mos

def parse_storey_range(val):
    if pd.isna(val): return (np.nan, np.nan)
    nums = re.findall(r"\d+", str(val))
    if len(nums) == 1: return (int(nums[0]), int(nums[0]))
    if len(nums) >= 2: return (int(nums[0]), int(nums[1]))
    return (np.nan, np.nan)

# ---------- Time starts ----------
def add_time_start_columns(out: pd.DataFrame):
    if "month" in out.columns:
        out["month_start"] = pd.to_datetime(out["month"], format="%Y-%m", errors="coerce") + MonthBegin(0)
    if {"year", "quarter_num"}.issubset(out.columns):
        out["quarter_start"] = pd.to_datetime(
            out["year"].astype(str) + "-" + ((out["quarter_num"] - 1) * 3 + 1).astype(str) + "-01",
            errors="coerce"
        )
    return out

# ---------- Enum asserts ----------
def assert_enums(df, col, allowed: set[str]):
    if col not in df.columns: return
    vals = set(df[col].dropna().unique())
    bad = vals - allowed
    if bad:
        raise ValueError(f"Unexpected values in {col}: {sorted(bad)} (allowed: {sorted(allowed)})")

ALLOWED_FLAT_TYPES = {"1 ROOM","2 ROOM","3 ROOM","4 ROOM","5 ROOM","EXECUTIVE","MULTI-GENERATION"}
ALLOWED_SEGMENTS   = {"Core Central Region","Rest of Central Region","Outside Central Region"}
ALLOWED_PROPERTY_TYPES = {"All Residential","Landed","Non-Landed"}

# ---------- HDB towns ----------
HDB_TOWN_MAP = {
    "ANG MO KIO":"Ang Mo Kio","BEDOK":"Bedok","BISHAN":"Bishan","BUKIT BATOK":"Bukit Batok",
    "BUKIT MERAH":"Bukit Merah","BUKIT PANJANG":"Bukit Panjang","BUKIT TIMAH":"Bukit Timah",
    "CENTRAL AREA":"Central Area","CHOA CHU KANG":"Choa Chu Kang","CLEMENTI":"Clementi",
    "GEYLANG":"Geylang","HOUGANG":"Hougang","JURONG EAST":"Jurong East","JURONG WEST":"Jurong West",
    "KALLANG/WHAMPOA":"Kallang/Whampoa","MARINE PARADE":"Marine Parade","PASIR RIS":"Pasir Ris",
    "PUNGGOL":"Punggol","QUEENSTOWN":"Queenstown","SEMBAWANG":"Sembawang","SENGKANG":"Sengkang",
    "SERANGOON":"Serangoon","TAMPINES":"Tampines","TOA PAYOH":"Toa Payoh","WOODLANDS":"Woodlands",
    "YISHUN":"Yishun"
}
def normalize_hdb_town(val):
    if pd.isna(val): return val
    key = str(val).strip().upper()
    return HDB_TOWN_MAP.get(key, str(val).strip().title())

# ---------- Sanity caps & currency ----------
def apply_sanity_caps(df, caps: dict[str, tuple[float,float]]):
    for col, (minv, maxv) in caps.items():
        if col in df.columns:
            x = pd.to_numeric(df[col], errors="coerce")
            df = df[(x.isna()) | ((x >= minv) & (x <= maxv))].copy()
    return df

# ---------- PK sorting ----------
def sort_by_pk(df, pk_cols: list[str]):
    existing = [c for c in pk_cols if c in df.columns]
    if existing:
        return df.sort_values(existing).reset_index(drop=True)
    return df

PK_HDB_RESALE = ["month","town","block","street_name","flat_type","floor_area_sqm","storey_range","lease_commence_date","resale_price"]
PK_QUARTERLY  = ["year","quarter_num"]  # extend with categoricals when available

# ---------- Metadata files ----------
def save_note(note_path: Path, info: dict):
    text = "PropInsight preprocess note (2023–2025 only)\n" \
           "-----------------------------------------------\n" + \
           "\n".join([f"{k}: {v}" for k, v in info.items()])
    note_path.write_text(text, encoding="utf-8")

def export_schema_and_counts(out: pd.DataFrame, base_path: str):
    schema = {c: str(out[c].dtype) for c in out.columns}
    counts = {c: {"distinct": int(out[c].nunique()), "null_pct": float(out[c].isna().mean())} for c in out.columns}
    Path(f"{base_path}_SCHEMA.json").write_text(json.dumps(schema, indent=2))
    Path(f"{base_path}_COUNTS.json").write_text(json.dumps(counts, indent=2))

# ---------- Common dedup + sanity ----------
def postprocess_common(df: pd.DataFrame, essential_cols: list[str], numeric_must_be_positive: list[str] = None):
    if numeric_must_be_positive is None:
        numeric_must_be_positive = []
    for c in df.columns:
        if pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip()
    df = df.dropna(subset=essential_cols).copy()
    for c in numeric_must_be_positive:
        if c in df.columns:
            df = df[(pd.to_numeric(df[c], errors="coerce") > 0)].copy()
    df = df.drop_duplicates().reset_index(drop=True)
    return df



#Specific handlers (with postprocess_common integrated)

In [ ]:
# @title HDB Resale (monthly) — preprocess 2023–2025 only
def process_hdb_resale_2023_2025(in_path: Path, out_dir: Path):
    out_path = out_dir / "HDB_Resale_2023_2025.csv"
    note_path = out_dir / "HDB_Resale_2023_2025_NOTE.txt"

    df_raw = pd.read_csv(in_path, dtype=str)
    df = normalize_columns(df_raw)

    if "month" not in df.columns or "resale_price" not in df.columns:
        raise ValueError(f"{in_path.name}: expected 'month' and 'resale_price' columns.")

    # Filter by month range
    df["month"] = df["month"].astype(str).str.strip()
    df["month_dt"] = pd.to_datetime(df["month"], format="%Y-%m", errors="coerce")
    df = df.dropna(subset=["month_dt"]).copy()
    df = df[(df["month_dt"] >= "2023-01-01") & (df["month_dt"] <= "2025-12-31")]

    # Numerics
    df["resale_price"] = coerce_numeric(df["resale_price"], allow_float=True)
    if "floor_area_sqm" in df.columns:
        df["floor_area_sqm"] = coerce_numeric(df["floor_area_sqm"], allow_float=True)
    if "lease_commence_date" in df.columns:
        df["lease_commence_date"] = coerce_numeric(df["lease_commence_date"], allow_float=False)

    # Text normalization
    def title_or_same(x):
        if pd.isna(x): return x
        s = str(x).strip()
        return " ".join([w.upper() if w.upper() in {"HDB","A","B","C","D","E"} else w.title() for w in s.split()])

    for col in ("town","street_name","flat_model"):
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().apply(title_or_same)
    if "flat_type" in df.columns:
        df["flat_type"] = df["flat_type"].astype(str).str.strip().str.upper()
    if "storey_range" in df.columns:
        df["storey_range"] = df["storey_range"].astype(str).str.strip().str.upper()
    if "block" in df.columns:
        df["block"] = df["block"].astype(str).str.strip().str.upper()

    # Select/order columns
    ordered = [
        "month", "town", "flat_type", "block", "street_name", "storey_range",
        "floor_area_sqm", "flat_model", "lease_commence_date",
        "remaining_lease", "resale_price"
    ]
    keep_cols = [c for c in ordered if c in df.columns]
    out = df[keep_cols].copy().reset_index(drop=True)

    # HDB Town normalization
    if "town" in out.columns:
        out["town"] = out["town"].apply(normalize_hdb_town)

    # Sanity caps + currency
    out = apply_sanity_caps(out, {
        "floor_area_sqm": (10, 400),
        "resale_price": (50_000, 2_500_000)
    })
    if "resale_price" in out.columns and "currency" not in out.columns:
        out["currency"] = "SGD"

    # Parse lease & storey
    if "remaining_lease" in out.columns:
        out["remaining_lease_months"] = out["remaining_lease"].apply(parse_remaining_lease_to_months)
    if "storey_range" in out.columns:
        parsed = out["storey_range"].apply(parse_storey_range)
        out["storey_low"]  = parsed.apply(lambda x: x[0])
        out["storey_high"] = parsed.apply(lambda x: x[1])

    # Enum assert (flat types)
    if "flat_type" in out.columns:
        assert_enums(out, "flat_type", ALLOWED_FLAT_TYPES)

    # Common postprocess
    out = postprocess_common(
        out,
        essential_cols=["month", "resale_price"],
        numeric_must_be_positive=["resale_price"] + (["floor_area_sqm"] if "floor_area_sqm" in out.columns else [])
    )

    # Final cleanup, time starts, PK sort, metadata
    out = strip_invisible_chars(out)
    out = add_time_start_columns(out)
    out = sort_by_pk(out, PK_HDB_RESALE)

    out.to_csv(out_path, index=False)
    months = out["month"].dropna().astype(str).sort_values().unique().tolist()
    save_note(note_path, {
        "Source file": in_path.name,
        "Rows in source": len(df_raw),
        "Rows after filter (2023–2025)": len(out),
        "Distinct months (2023–2025)": len(months),
        "First month": months[0] if months else "N/A",
        "Last month": months[-1] if months else "N/A",
        "Schema saved": ", ".join(out.columns),
        "No analysis performed": True
    })
    export_schema_and_counts(out, str(out_path).replace(".csv",""))
    return out_path


#Private Property Price Index (Quarterly) — PrivateResidentialPropertyPriceIndexBaseQuarter2009Q1100Quarterly.csv

In [ ]:
# @title Private Property Price Index (quarterly) — preprocess 2023–2025 only
PROPERTY_TYPE_MAP = {
    "all residential": "All Residential",
    "landed": "Landed",
    "non-landed": "Non-Landed",
}

def process_price_index_quarterly(in_path: Path, out_dir: Path):
    out_path = out_dir / "PrivatePropertyPriceIndex_Quarterly_2023_2025.csv"
    note_path = out_dir / "PrivatePropertyPriceIndex_Quarterly_2023_2025_NOTE.txt"

    df_raw = pd.read_csv(in_path, dtype=str)
    df = normalize_columns(df_raw)

    colmap = {}
    for c in df.columns:
        lc = c.lower().strip()
        if lc == "quarter": colmap[c] = "quarter"
        if lc in ("property_type","property type","type"): colmap[c] = "property_type"
        if lc in ("index","price_index","ppi"): colmap[c] = "index"
    df = df.rename(columns=colmap)

    if not {"quarter","property_type","index"}.issubset(df.columns):
        raise ValueError(f"{in_path.name}: expected quarter, property_type, index")

    q = df["quarter"].apply(parse_quarter_str)
    df["quarter_norm"] = q.apply(lambda x: x[0])
    df["year"] = q.apply(lambda x: x[1]).astype("Int64")
    df["quarter_num"] = q.apply(lambda x: x[2]).astype("Int64")
    df = df.dropna(subset=["quarter_norm","year","quarter_num"]).copy()
    df = df[(df["year"]>=2023) & (df["year"]<=2025)]

    df["property_type"] = df["property_type"].astype(str).str.strip().str.lower().map(PROPERTY_TYPE_MAP).fillna(df["property_type"])
    df["index"] = coerce_numeric(df["index"], allow_float=True)

    out = df[["quarter_norm","year","quarter_num","property_type","index"]]\
            .dropna(subset=["index"])\
            .sort_values(["year","quarter_num","property_type"]).reset_index(drop=True)

    # Enum assert
    assert_enums(out, "property_type", ALLOWED_PROPERTY_TYPES)

    # Postprocess
    out = postprocess_common(out,
        essential_cols=["quarter_norm", "property_type", "index"],
        numeric_must_be_positive=["index"]
    )
    out = strip_invisible_chars(out)
    out = add_time_start_columns(out)
    out = sort_by_pk(out, PK_QUARTERLY + ["property_type"])

    out.to_csv(out_path, index=False)
    save_note(note_path, {
        "Source file": in_path.name,
        "Rows in source": len(df_raw),
        "Rows (2023–2025)": len(out),
        "Schema": ", ".join(out.columns),
        "No analysis performed": True
    })
    export_schema_and_counts(out, str(out_path).replace(".csv",""))
    return out_path


#Private Property Rental Index (Quarterly) — PrivateResidentialPropertyRentalIndexBaseQuarter2009Q1100Quarterly.csv

In [ ]:
# @title Private Property Rental Index (quarterly) — preprocess 2023–2025 only
def process_rental_index_quarterly(in_path: Path, out_dir: Path):
    out_path = out_dir / "PrivatePropertyRentalIndex_Quarterly_2023_2025.csv"
    note_path = out_dir / "PrivatePropertyRentalIndex_Quarterly_2023_2025_NOTE.txt"

    df_raw = pd.read_csv(in_path, dtype=str)
    df = normalize_columns(df_raw)

    colmap = {}
    for c in df.columns:
        lc = c.lower().strip()
        if lc == "quarter": colmap[c] = "quarter"
        if lc in ("property_type","property type","type"): colmap[c] = "property_type"
    df = df.rename(columns=colmap)

    if "quarter" not in df.columns or "index" not in df.columns:
        raise ValueError(f"{in_path.name}: expected quarter and index columns")

    q = df["quarter"].apply(parse_quarter_str)
    df["quarter_norm"] = q.apply(lambda x: x[0])
    df["year"] = q.apply(lambda x: x[1]).astype("Int64")
    df["quarter_num"] = q.apply(lambda x: x[2]).astype("Int64")
    df = df.dropna(subset=["quarter_norm","year","quarter_num"]).copy()
    df = df[(df["year"]>=2023) & (df["year"]<=2025)]

    cols = ["quarter_norm","year","quarter_num","index"]
    if "property_type" in df.columns:
        df["property_type"] = df["property_type"].astype(str).str.strip().str.lower().map(PROPERTY_TYPE_MAP).fillna(df["property_type"])
        cols = ["quarter_norm","year","quarter_num","property_type","index"]

    df["index"] = coerce_numeric(df["index"], allow_float=True)
    out = df[cols].dropna(subset=["index"]).sort_values(["year","quarter_num"]).reset_index(drop=True)

    # Enum assert (if present)
    if "property_type" in out.columns:
        assert_enums(out, "property_type", ALLOWED_PROPERTY_TYPES)

    # Postprocess
    out = postprocess_common(out,
        essential_cols=["quarter_norm", "index"],
        numeric_must_be_positive=["index"]
    )
    out = strip_invisible_chars(out)
    out = add_time_start_columns(out)
    sort_cols = PK_QUARTERLY + (["property_type"] if "property_type" in out.columns else [])
    out = sort_by_pk(out, sort_cols)

    out.to_csv(out_path, index=False)
    save_note(note_path, {
        "Source file": in_path.name,
        "Rows (2023–2025)": len(out),
        "Schema": ", ".join(out.columns),
        "No analysis performed": True
    })
    export_schema_and_counts(out, str(out_path).replace(".csv",""))
    return out_path


#Price Index (unsuffixed) — PrivateResidentialPropertyPriceIndexBaseQuarter2009Q1100.csv

In [ ]:
# @title Private Property Price Index (unsuffixed) — preprocess 2023–2025 only
def process_price_index_base(in_path: Path, out_dir: Path):
    df_raw = pd.read_csv(in_path, dtype=str)
    df = normalize_columns(df_raw)
    if "quarter" in df.columns:
        return process_price_index_quarterly(in_path, out_dir)

    quarters = detect_quarter_headers(df)
    out_path = out_dir / "PrivatePropertyPriceIndex_Base_2023_2025.csv"
    note_path = out_dir / "PrivatePropertyPriceIndex_Base_2023_2025_NOTE.txt"

    if not quarters:
        out = pd.DataFrame(columns=["quarter_norm","year","quarter_num","series","index"])
        out.to_csv(out_path, index=False)
        save_note(note_path, {"Source file": in_path.name, "Rows (2023–2025)": 0, "Info": "No quarter headers detected"})
        export_schema_and_counts(out, str(out_path).replace(".csv",""))
        return out_path

    keep = [x[0] for x in quarters if 2023 <= x[1] <= 2025]
    if not keep:
        out = pd.DataFrame(columns=["quarter_norm","year","quarter_num","series","index"])
        out.to_csv(out_path, index=False)
        save_note(note_path, {"Source file": in_path.name, "Rows (2023–2025)": 0, "Info": "No 2023–2025 quarter columns detected"})
        export_schema_and_counts(out, str(out_path).replace(".csv",""))
        return out_path

    id_col = None
    for c in df.columns:
        if c not in keep:
            id_col = c; break
    if id_col is None:
        id_col = "series"; df[id_col] = "Index"

    melted = df[[id_col] + keep].melt(id_vars=[id_col], var_name="quarter_col", value_name="index_raw")
    parsed = melted["quarter_col"].apply(lambda h: QUARTER_RE.match(str(h).strip()))
    melted["year"] = parsed.apply(lambda m: int(m.group("year")) if m else np.nan)
    melted["quarter_num"] = parsed.apply(lambda m: int(m.group("q")) if m else np.nan)
    melted = melted.dropna(subset=["year","quarter_num"])
    melted["year"] = melted["year"].astype(int); melted["quarter_num"] = melted["quarter_num"].astype(int)
    melted["quarter_norm"] = melted.apply(lambda r: f"{r['year']}-Q{r['quarter_num']}", axis=1)
    melted["index"] = coerce_numeric(melted["index_raw"], allow_float=True)

    out = melted[["quarter_norm","year","quarter_num",id_col,"index"]]\
          .dropna(subset=["index"]).sort_values(["year","quarter_num",id_col]).reset_index(drop=True)

    out = postprocess_common(out, essential_cols=["quarter_norm","index"], numeric_must_be_positive=["index"])
    out = strip_invisible_chars(out)
    out = add_time_start_columns(out)
    out = sort_by_pk(out, PK_QUARTERLY + [id_col])

    out.to_csv(out_path, index=False)
    save_note(note_path, {"Source file": in_path.name, "Rows (2023–2025)": len(out), "Schema": ", ".join(out.columns), "No analysis performed": True})
    export_schema_and_counts(out, str(out_path).replace(".csv",""))
    return out_path


#Regional Transactions (Quarterly long/wide) — CCR/OCR/Whole SG

In [ ]:
# @title Private Residential Transactions (regional quarterly) — preprocess 2023–2025 only
def infer_region_from_name(name: str) -> str:
    n = name.lower()
    if "corecentralregion" in n: return "Core Central Region"
    if "outsidecentralregion" in n: return "Outside Central Region"
    if "wholeofsingapore" in n: return "Whole of Singapore"
    return "Unknown"

def process_transactions_quarterly(in_path: Path, out_dir: Path):
    region = infer_region_from_name(in_path.name)
    out_path = out_dir / f"PrivateTransactions_{region.replace(' ', '')}_Quarterly_2023_2025.csv"
    note_path = out_dir / f"PrivateTransactions_{region.replace(' ', '')}_Quarterly_2023_2025_NOTE.txt"

    df_raw = pd.read_csv(in_path, dtype=str)
    df = normalize_columns(df_raw)

    if "quarter" in df.columns:
        q = df["quarter"].apply(parse_quarter_str)
        df["quarter_norm"] = q.apply(lambda x: x[0])
        df["year"] = q.apply(lambda x: x[1]).astype("Int64")
        df["quarter_num"] = q.apply(lambda x: x[2]).astype("Int64")
        df = df.dropna(subset=["quarter_norm","year","quarter_num"]).copy()
        df = df[(df["year"]>=2023) & (df["year"]<=2025)]
        df["region"] = region

        # Best-effort numeric coercion
        for c in df.columns:
            if c not in ("quarter","quarter_norm","year","quarter_num","region"):
                tmp = coerce_numeric(df[c], allow_float=True)
                if tmp.notna().sum() >= (0.5 * len(tmp)):
                    df[c] = tmp

        out_cols_front = ["quarter_norm","year","quarter_num","region"]
        other_cols = [c for c in df.columns if c not in set(out_cols_front + ["quarter"])]
        out = df[out_cols_front + other_cols].reset_index(drop=True).sort_values(["year","quarter_num"])

        out = postprocess_common(out, essential_cols=["quarter_norm","year","quarter_num"], numeric_must_be_positive=[])
        out = strip_invisible_chars(out)
        out = add_time_start_columns(out)
        out = sort_by_pk(out, PK_QUARTERLY + ["region"])

        out.to_csv(out_path, index=False)
        save_note(note_path, {"Source file": in_path.name, "Rows (2023–2025)": len(out), "Region": region, "Schema": ", ".join(out.columns), "No analysis performed": True})
        export_schema_and_counts(out, str(out_path).replace(".csv",""))
        return out_path

    # wide → melt
    quarters = detect_quarter_headers(df)
    if quarters:
        keep = [x[0] for x in quarters if 2023 <= x[1] <= 2025]
        id_col = None
        for c in df.columns:
            if c not in keep:
                id_col = c; break
        if id_col is None:
            id_col = "series"; df[id_col] = "Transactions"

        melted = df[[id_col] + keep].melt(id_vars=[id_col], var_name="quarter_col", value_name="value_raw")
        parsed = melted["quarter_col"].apply(lambda h: QUARTER_RE.match(str(h).strip()))
        melted["year"] = parsed.apply(lambda m: int(m.group("year")) if m else np.nan)
        melted["quarter_num"] = parsed.apply(lambda m: int(m.group("q")) if m else np.nan)
        melted = melted.dropna(subset=["year","quarter_num"])
        melted["year"] = melted["year"].astype(int); melted["quarter_num"] = melted["quarter_num"].astype(int)
        melted["quarter_norm"] = melted.apply(lambda r: f"{r['year']}-Q{r['quarter_num']}", axis=1)
        melted["region"] = region
        melted["value"] = coerce_numeric(melted["value_raw"], allow_float=True)

        out = melted[["quarter_norm","year","quarter_num","region",id_col,"value"]]\
              .dropna(subset=["value"]).sort_values(["year","quarter_num",id_col]).reset_index(drop=True)

        out = postprocess_common(out, essential_cols=["quarter_norm","year","quarter_num","value"], numeric_must_be_positive=["value"])
        out = strip_invisible_chars(out)
        out = add_time_start_columns(out)
        out = sort_by_pk(out, PK_QUARTERLY + ["region", id_col])

        out.to_csv(out_path, index=False)
        save_note(note_path, {"Source file": in_path.name, "Rows (2023–2025)": len(out), "Region": region, "Schema": ", ".join(out.columns), "No analysis performed": True})
        export_schema_and_counts(out, str(out_path).replace(".csv",""))
        return out_path

    raise ValueError(f"{in_path.name}: couldn't detect a 'quarter' column or quarter headers")


#Uncompleted Units — UncompletedPrivateResidentialUnitsLaunchedintheQuarterbyMarketSegmentQuarterly.csv

In [ ]:
# @title Uncompleted Private Residential Units Launched (quarterly) — preprocess 2023–2025 only
SEGMENT_MAP = {
    "core central region":"Core Central Region",
    "rest of central region":"Rest of Central Region",
    "outside central region":"Outside Central Region",
    "ccr":"Core Central Region","rcr":"Rest of Central Region","ocr":"Outside Central Region",
}
def norm_segment(x):
    if pd.isna(x): return None
    s = str(x).strip().lower()
    return SEGMENT_MAP.get(s, s.title())

def process_uncompleted_units(in_path: Path, out_dir: Path):
    out_path = out_dir / "UncompletedPrivateResidentialUnitsLaunched_2023_2025.csv"
    note_path = out_dir / "UncompletedPrivateResidentialUnitsLaunched_2023_2025_NOTE.txt"

    df_raw = pd.read_csv(in_path, dtype=str)
    df = normalize_columns(df_raw)

    colmap = {}
    for c in df.columns:
        lc = c.strip().lower()
        if lc == "quarter": colmap[c] = "quarter"
        if lc in ("market_segment","market segment","segment"): colmap[c] = "market_segment"
        if lc in ("units","no_of_units","no. of units","number_of_units"): colmap[c] = "units"
    df = df.rename(columns=colmap)

    if not {"quarter","market_segment","units"}.issubset(df.columns):
        raise ValueError(f"{in_path.name}: expected quarter, market_segment, units")

    q_parsed = df["quarter"].apply(parse_quarter_str)
    df["quarter_norm"] = q_parsed.apply(lambda x: x[0])
    df["year"] = q_parsed.apply(lambda x: x[1]).astype("Int64")
    df["quarter_num"] = q_parsed.apply(lambda x: x[2]).astype("Int64")

    df["market_segment"] = df["market_segment"].apply(norm_segment)
    df["units"] = coerce_numeric(df["units"], allow_float=True)

    out = df.dropna(subset=["quarter_norm","year","quarter_num","units","market_segment"]).copy()
    out = out[(out["year"]>=2023) & (out["year"]<=2025)]
    out = out[["quarter_norm","year","quarter_num","market_segment","units"]]\
             .sort_values(["year","quarter_num","market_segment"]).reset_index(drop=True)

    # Enum assert
    assert_enums(out, "market_segment", ALLOWED_SEGMENTS)

    out = postprocess_common(out, essential_cols=["quarter_norm","year","quarter_num","units"], numeric_must_be_positive=["units"])
    out = strip_invisible_chars(out)
    out = add_time_start_columns(out)
    out = sort_by_pk(out, PK_QUARTERLY + ["market_segment"])

    out.to_csv(out_path, index=False)
    save_note(note_path, {"Source file": in_path.name, "Rows (2023–2025)": len(out), "Schema": ", ".join(out.columns), "No analysis performed": True})
    export_schema_and_counts(out, str(out_path).replace(".csv",""))
    return out_path


#Supply Pipeline — SupplyOfCommercialAndIndustrialPropertiesInThePipelineByDevelopmentStatusPrivateAndPublicSectorsEndOfPeriodQuarterly.csv

In [ ]:
# @title Supply Pipeline (quarterly wide → tidy) — preprocess 2023–2025 only
PROPERTY_NORMALIZE = {
    "total office space": "Office Space",
    "total business park space": "Business Park Space",
    "total multiple-user factory space": "Multiple-User Factory Space",
    "total single-user factory space": "Single-User Factory Space",
    "total warehouse space": "Warehouse Space",
    "total retail space": "Retail Space",
    "total hotel rooms": "Hotel Rooms",
}
STATUS_NORMALIZE = {
    "under construction": "Under Construction",
    "planned - written permission": "Planned - Written Permission",
    "planned - provisional permission": "Planned - Provisional Permission",
    "planned - others": "Planned - Others",
}
SECTOR_NORMALIZE = {
    "private sector": "Private Sector",
    "public sector": "Public Sector",
    "private and public sectors": "Private and Public Sectors",
    "all sectors": "All Sectors",
}

def parse_dataseries(raw):
    if pd.isna(raw): return (None,None,None)
    parts = [p.strip() for p in str(raw).split(" - ") if p.strip()]
    prop, status, sector = None, None, None
    if parts:
        prop = PROPERTY_NORMALIZE.get(parts[0].lower(), parts[0])
    if len(parts) >= 2:
        p2 = parts[1].lower()
        if p2.startswith("planned"):
            if len(parts) >= 3:
                status = STATUS_NORMALIZE.get(f"planned - {parts[2].lower()}", f"Planned - {parts[2]}")
                if len(parts) >= 4:
                    sector = SECTOR_NORMALIZE.get(parts[3].lower(), parts[3])
            else:
                status = STATUS_NORMALIZE.get("planned - others", "Planned - Others")
        elif "under construction" in p2:
            status = STATUS_NORMALIZE["under construction"]
            if len(parts) >= 3:
                sector = SECTOR_NORMALIZE.get(parts[2].lower(), parts[2])
        else:
            sector = SECTOR_NORMALIZE.get(parts[1].lower(), parts[1])
    if sector is None: sector = "All Sectors"
    return (prop, status, sector)

def process_supply_pipeline(in_path: Path, out_dir: Path):
    out_path = out_dir / "SupplyPipeline_2023_2025.csv"
    note_path = out_dir / "SupplyPipeline_2023_2025_NOTE.txt"

    df_raw = pd.read_csv(in_path, dtype=str)
    df = normalize_columns(df_raw)
    if "DataSeries" not in df.columns:
        for c in df.columns:
            if c.lower().replace(" ","") in ("dataseries","series","indicator","item"):
                df = df.rename(columns={c:"DataSeries"})
                break
    if "DataSeries" not in df.columns:
        raise ValueError(f"{in_path.name}: expected DataSeries column")

    quarters = detect_quarter_headers(df)
    if not quarters:
        out = pd.DataFrame(columns=["quarter_norm","year","quarter_num","property_type","development_status","sector","units"])
        out.to_csv(out_path, index=False)
        save_note(note_path, {"Source file": in_path.name, "Rows (2023–2025)": 0, "Info": "No quarter headers detected"})
        export_schema_and_counts(out, str(out_path).replace(".csv",""))
        return out_path

    keep = [x[0] for x in quarters if 2023 <= x[1] <= 2025]
    if not keep:
        out = pd.DataFrame(columns=["quarter_norm","year","quarter_num","property_type","development_status","sector","units"])
        out.to_csv(out_path, index=False)
        save_note(note_path, {"Source file": in_path.name, "Rows (2023–2025)": 0, "Info": "No 2023–2025 quarter columns detected"})
        export_schema_and_counts(out, str(out_path).replace(".csv",""))
        return out_path

    d = df[["DataSeries"] + keep].copy()
    long = d.melt(id_vars=["DataSeries"], var_name="quarter_col", value_name="units_raw")

    parsed = long["quarter_col"].apply(lambda h: QUARTER_RE.match(str(h).strip()))
    long["year"] = parsed.apply(lambda m: int(m.group("year")) if m else np.nan)
    long["quarter_num"] = parsed.apply(lambda m: int(m.group("q")) if m else np.nan)
    long = long.dropna(subset=["year","quarter_num"])
    long["year"] = long["year"].astype(int); long["quarter_num"] = long["quarter_num"].astype(int)
    long["quarter_norm"] = long.apply(lambda r: f"{r['year']}-Q{r['quarter_num']}", axis=1)
    long["units"] = coerce_numeric(long["units_raw"], allow_float=True)

    parsed_series = long["DataSeries"].apply(parse_dataseries)
    long["property_type"] = parsed_series.apply(lambda x: x[0])
    long["development_status"] = parsed_series.apply(lambda x: x[1])
    long["sector"] = parsed_series.apply(lambda x: x[2])

    out = long[["quarter_norm","year","quarter_num","property_type","development_status","sector","units"]]\
          .dropna(subset=["quarter_norm","year","quarter_num","property_type","units"])\
          .sort_values(["year","quarter_num","property_type","development_status","sector"]).reset_index(drop=True)

    out = postprocess_common(out, essential_cols=["quarter_norm","year","quarter_num","units"], numeric_must_be_positive=["units"])
    out = strip_invisible_chars(out)
    out = add_time_start_columns(out)
    out = sort_by_pk(out, PK_QUARTERLY + ["property_type","development_status","sector"])

    out.to_csv(out_path, index=False)
    save_note(note_path, {"Source file": in_path.name, "Rows (2023–2025)": len(out), "Schema": ", ".join(out.columns), "No analysis performed": True})
    export_schema_and_counts(out, str(out_path).replace(".csv",""))
    return out_path


#Bookings for New Flats — BookingsforNewFlats.csv

In [ ]:
# @title Bookings for New Flats (yearly) — preprocess 2023–2025 only
def process_bookings_new_flats(in_path: Path, out_dir: Path):
    out_path = out_dir / "BookingsforNewFlats_2023_2025.csv"
    note_path = out_dir / "BookingsforNewFlats_2023_2025_NOTE.txt"

    df_raw = pd.read_csv(in_path, dtype=str)
    df = normalize_columns(df_raw)
    rename_map = {}
    for c in df.columns:
        lc = c.strip().lower()
        if lc == "financial_year": rename_map[c] = "financial_year"
        if lc in ("no_of_units","no. of units","units","number_of_units"): rename_map[c] = "no_of_units"
    df = df.rename(columns=rename_map)

    if not {"financial_year","no_of_units"}.issubset(df.columns):
        raise ValueError(f"{in_path.name}: expected financial_year, no_of_units")

    df["financial_year"] = pd.to_numeric(df["financial_year"], errors="coerce")
    df["no_of_units"] = coerce_numeric(df["no_of_units"], allow_float=True)
    out = df[(df["financial_year"]>=2023) & (df["financial_year"]<=2025)].copy()
    out = out[["financial_year","no_of_units"]].sort_values("financial_year").reset_index(drop=True)

    out = postprocess_common(out, essential_cols=["financial_year","no_of_units"], numeric_must_be_positive=["no_of_units"])
    out = strip_invisible_chars(out)
    # (yearly → no month/quarter columns)
    out = sort_by_pk(out, ["financial_year"])

    out.to_csv(out_path, index=False)
    save_note(note_path, {"Source file": in_path.name, "Rows (2023–2025)": len(out), "Schema": ", ".join(out.columns), "No analysis performed": True})
    export_schema_and_counts(out, str(out_path).replace(".csv",""))
    return out_path


#Router & Batch Runner (process all CSVs in raw/data_gov)

In [ ]:
# @title Router & Batch Runner
from typing import Callable
import os

ROUTES: list[tuple[str, Callable]] = [
    ("ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv", process_hdb_resale_2023_2025),
    ("PrivateResidentialPropertyPriceIndexBaseQuarter2009Q1100Quarterly.csv", process_price_index_quarterly),
    ("PrivateResidentialPropertyRentalIndexBaseQuarter2009Q1100Quarterly.csv", process_rental_index_quarterly),
    ("PrivateResidentialPropertyPriceIndexBaseQuarter2009Q1100.csv", process_price_index_base),
    ("PrivateResidentialPropertyTransactionsinCoreCentralRegionQuarterly.csv", process_transactions_quarterly),
    ("PrivateResidentialPropertyTransactionsinOutsideCentralRegionQuarterly.csv", process_transactions_quarterly),
    ("PrivateResidentialPropertyTransactionsintheWholeofSingaporeQuarterly.csv", process_transactions_quarterly),
    ("UncompletedPrivateResidentialUnitsLaunchedintheQuarterbyMarketSegmentQuarterly.csv", process_uncompleted_units),
    ("SupplyOfCommercialAndIndustrialPropertiesInThePipelineByDevelopmentStatusPrivateAndPublicSectorsEndOfPeriodQuarterly.csv", process_supply_pipeline),
    ("BookingsforNewFlats.csv", process_bookings_new_flats),
]

def choose_processor(file_path: Path) -> Callable:
    name = file_path.name
    for pattern, func in ROUTES:
        if name == pattern:
            return func
    # Heuristics for unknown files
    try:
        sample = pd.read_csv(file_path, nrows=1, dtype=str)
        cols_lower = [c.strip().lower() for c in sample.columns]
        if "quarter" in cols_lower:
            return lambda p, o: fallback_quarter_long(p, o)
        if detect_quarter_headers(sample):
            return lambda p, o: fallback_quarter_wide(p, o)
    except Exception:
        pass
    return lambda p, o: fallback_quarter_long(p, o)

# Fallbacks
def fallback_quarter_long(in_path: Path, out_dir: Path):
    out_path = out_dir / f"{in_path.stem}_2023_2025.csv"
    note_path = out_dir / f"{in_path.stem}_2023_2025_NOTE.txt"

    df_raw = pd.read_csv(in_path, dtype=str)
    df = normalize_columns(df_raw)
    if "quarter" not in df.columns:
        raise ValueError("no 'quarter' column")

    q = df["quarter"].apply(parse_quarter_str)
    df["quarter_norm"] = q.apply(lambda x: x[0]); df["year"] = q.apply(lambda x: x[1]); df["quarter_num"] = q.apply(lambda x: x[2])
    df = df.dropna(subset=["quarter_norm","year","quarter_num"]).copy()
    df["year"] = df["year"].astype(int); df["quarter_num"] = df["quarter_num"].astype(int)
    df = df[(df["year"]>=2023) & (df["year"]<=2025)]

    # Best-effort numeric coercion
    for c in df.columns:
        if c not in ("quarter","quarter_norm","year","quarter_num"):
            tmp = coerce_numeric(df[c], allow_float=True)
            if tmp.notna().sum() >= (0.5 * len(tmp)):
                df[c] = tmp

    out_cols_front = ["quarter_norm","year","quarter_num"]
    other_cols = [c for c in df.columns if c not in set(out_cols_front + ["quarter"])]
    out = df[out_cols_front + other_cols].reset_index(drop=True).sort_values(["year","quarter_num"])

    out = postprocess_common(out, essential_cols=["quarter_norm","year","quarter_num"])
    out = strip_invisible_chars(out)
    out = add_time_start_columns(out)
    out = sort_by_pk(out, PK_QUARTERLY + [c for c in other_cols if c in df.columns])

    out.to_csv(out_path, index=False)
    save_note(note_path, {"Source file": in_path.name, "Rows (2023–2025)": len(out), "Schema": ", ".join(out.columns), "No analysis performed": True})
    export_schema_and_counts(out, str(out_path).replace(".csv",""))
    return out_path

def fallback_quarter_wide(in_path: Path, out_dir: Path):
    out_path = out_dir / f"{in_path.stem}_2023_2025.csv"
    note_path = out_dir / f"{in_path.stem}_2023_2025_NOTE.txt"

    df_raw = pd.read_csv(in_path, dtype=str)
    df = normalize_columns(df_raw)
    quarters = detect_quarter_headers(df)
    if not quarters:
        raise ValueError("no quarter headers like '2024Q2' / '2024-Q2'")

    keep = [x[0] for x in quarters if 2023 <= x[1] <= 2025]
    if not keep:
        out = pd.DataFrame(columns=["quarter_norm","year","quarter_num","series","value"])
        out.to_csv(out_path, index=False)
        save_note(note_path, {"Source file": in_path.name, "Rows (2023–2025)": 0, "Info": "No 2023–2025 quarter columns detected"})
        export_schema_and_counts(out, str(out_path).replace(".csv",""))
        return out_path

    # pick id column
    id_col = None
    for c in df.columns:
        if c not in keep:
            id_col = c; break
    if id_col is None:
        id_col = "series"; df[id_col] = "Value"

    melted = df[[id_col] + keep].melt(id_vars=[id_col], var_name="quarter_col", value_name="value_raw")
    parsed = melted["quarter_col"].apply(lambda h: QUARTER_RE.match(str(h).strip()))
    melted["year"] = parsed.apply(lambda m: int(m.group("year")) if m else np.nan)
    melted["quarter_num"] = parsed.apply(lambda m: int(m.group("q")) if m else np.nan)
    melted = melted.dropna(subset=["year","quarter_num"])
    melted["year"] = melted["year"].astype(int); melted["quarter_num"] = melted["quarter_num"].astype(int)
    melted["quarter_norm"] = melted.apply(lambda r: f"{r['year']}-Q{r['quarter_num']}", axis=1)
    melted["value"] = coerce_numeric(melted["value_raw"], allow_float=True)

    out = melted[["quarter_norm","year","quarter_num",id_col,"value"]]\
          .dropna(subset=["value"]).sort_values(["year","quarter_num",id_col]).reset_index(drop=True)

    out = postprocess_common(out, essential_cols=["quarter_norm","year","quarter_num","value"], numeric_must_be_positive=["value"])
    out = strip_invisible_chars(out)
    out = add_time_start_columns(out)
    out = sort_by_pk(out, PK_QUARTERLY + [id_col])

    out.to_csv(out_path, index=False)
    save_note(note_path, {"Source file": in_path.name, "Rows (2023–2025)": len(out), "Schema": f"quarter_norm, year, quarter_num, {id_col}, value", "No analysis performed": True})
    export_schema_and_counts(out, str(out_path).replace(".csv",""))
    return out_path

# --------- Run over all CSVs ----------
outputs = []
for csv_path in sorted(RAW_DIR.glob("*.csv")):
    try:
        proc = choose_processor(csv_path)
        out = proc(csv_path, PREP_DIR)
        outputs.append({"file": csv_path.name, "output": str(out)})
        print("✅ Processed:", csv_path.name, "->", Path(out).name)
    except FileNotFoundError:
        print(f"❌ Skipped: {csv_path.name} - File not found. Please check the file path and name.")
    except Exception as e:
        print("❌ Skipped with error:", csv_path.name, "-", e)

print("\nSummary:")
print(json.dumps(outputs, indent=2))

✅ Processed: BookingsforNewFlats.csv -> BookingsforNewFlats_2023_2025.csv
❌ Skipped with error: DemandforRentalandSoldFlats.csv - no 'quarter' column
✅ Processed: ExecutiveCondominiumUnitsLaunchedandSoldintheQuarterQuarterly.csv -> ExecutiveCondominiumUnitsLaunchedandSoldintheQuarterQuarterly_2023_2025.csv
❌ Skipped with error: PrivateResidentialPropertyPriceIndexBaseQuarter2009Q1100.csv - PrivateResidentialPropertyPriceIndexBaseQuarter2009Q1100.csv: expected quarter, property_type, index
✅ Processed: PrivateResidentialPropertyPriceIndexBaseQuarter2009Q1100Quarterly.csv -> PrivatePropertyPriceIndex_Quarterly_2023_2025.csv
✅ Processed: PrivateResidentialPropertyRentalIndexBaseQuarter2009Q1100Quarterly.csv -> PrivatePropertyRentalIndex_Quarterly_2023_2025.csv
✅ Processed: PrivateResidentialPropertyTransactionsinCoreCentralRegionQuarterly.csv -> PrivateTransactions_CoreCentralRegion_Quarterly_2023_2025.csv
✅ Processed: PrivateResidentialPropertyTransactionsinOutsideCentralRegionQuarterly.